# `sparsegf2.circuits.runner`: execute a realization, read the order parameter

[`runner.py`](../../src/sparsegf2/circuits/runner.py) is the thin layer that
walks a `CircuitBuilder`'s layer stream on a `SparseGF2` instance, then reads
the picture's order parameter into a `SampleRecord`. It is where the
[`scheduler`](scheduler.ipynb), [`picture`](picture.ipynb), and
[`clifford_table`](clifford_table.ipynb) pieces meet the core simulator.

## Contents
1. `simulate` and `SimulationRunner`
2. The **two RNG streams** (construction vs. outcome) and the independence trick
3. The run loop: warmup, gates, measurements
4. Observables: half-cut entropy and code dimension
5. The `SampleRecord` union shape
6. Reproducibility, injected tables, `save_tableau`

## 1. `simulate` and `SimulationRunner`

```python
def simulate(config, *, sample_seed=0, save_tableau=False) -> SampleRecord:
    return SimulationRunner(config).run(sample_seed, save_tableau=save_tableau)
```

`simulate` is the one-liner entry point. `SimulationRunner` lets you build
once and run many seeds, sharing the (process-cached) $\mathrm{Sp}(4)$ table;
the table is also **injectable** for tests or custom gate sets.

In [1]:
from sparsegf2.circuits import CircuitConfig, Picture, simulate
cfg = CircuitConfig(graph_spec='cycle', n=8, picture=Picture.PURE_STATE,
                    gating_mode='brickwork', matching_mode='round_robin',
                    measurement_mode='bernoulli', p=0.16, depth_factor=4)
rec = simulate(cfg, sample_seed=0)
print('total_layers       :', rec.total_layers)       # depth_factor*n = 32
print('total_gates        :', rec.total_gates)         # 32 * n/2 = 128
print('total_measurements :', rec.total_measurements)
print('entropy_half_cut   :', rec.entropy_half_cut)    # <= n/2
print('code_dimension     :', rec.code_dimension)      # None for pure_state
print('ratio exp/act      :', round(rec.gate_to_meas_ratio_expected,3),
      '/', round(rec.gate_to_meas_ratio_actual,3))


total_layers       : 32
total_gates        : 128
total_measurements : 51
entropy_half_cut   : 3
code_dimension     : None
ratio exp/act      : 3.125 / 2.51


## 2. Two RNG streams, and why they're independent

A run uses **two** generators, by design:

- **Circuit construction**: the `CircuitBuilder`, seeded `base_seed +
  sample_seed`. Decides gate pairs, Clifford indices, measured qubits.
- **Measurement outcomes**: lives on the `SparseGF2` instance, seeded
  *independently* in the runner as
  ```python
  np.random.default_rng([base_seed + sample_seed, 0x6D656173])   # "meas"
  ```

Why the two-element seed? A plain integer seed `base+sample` is what the
*builder* uses. If the outcome stream were seeded `base+sample+2` (a tempting
"offset"), it would **collide** across samples: sample 0's outcome seed
(`base+2`) would equal sample 2's *construction* seed (`base+2`), correlating
the two. A **two-element** seed `[base+sample, KEY]` goes through numpy's
`SeedSequence` hash and shares entropy with no plain integer seed, so every
sample's two streams are mutually independent, and independent across
samples. Empirically the first draws are uncorrelated:

In [2]:
import numpy as np
KEY = 0x6D656173
# construction seed vs outcome seed for the same sample: different streams
for s in (0, 1, 2):
    constr = np.random.default_rng(42 + s).random()
    outcome = np.random.default_rng([42 + s, KEY]).random()
    print(f'sample {s}: construction first-draw={constr:.4f}  outcome first-draw={outcome:.4f}')
# no collision: outcome seed [42+s, KEY] never equals any plain int seed 42+s'
print('two-element seeds never collide with the builder\'s int seeds.')


sample 0: construction first-draw=0.7740  outcome first-draw=0.3706
sample 1: construction first-draw=0.6523  outcome first-draw=0.3069
sample 2: construction first-draw=0.1226  outcome first-draw=0.6003
two-element seeds never collide with the builder's int seeds.


## 3. The run loop

```python
sim_rng = np.random.default_rng([base_seed + sample_seed, _MEAS_STREAM_KEY])
sim, spec = setup_picture(picture, n, rng=sim_rng, pivot_mode=..., use_numba=...)
builder = CircuitBuilder(config, sample_seed)

for wlayer in builder.warmup_layers_iter():          # gate-only pre-scramble
    for (qi, qj), ci in ...: sim.apply_gate_2q(qi, qj, table[ci])

for layer in builder.layers():                       # measured phase
    for (qi, qj), ci in ...: sim.apply_gate_2q(qi, qj, table[ci])
    for q in layer.meas_qubits: sim.measure_z(q)      # outcome <- sim_rng
```

The Clifford index is taken `% len(table)`, a harmless safety net (indices
are already in `[0, n_cliffords) ⊆ [0, len(table))`). `measure_z` draws its
phase-free coin from `sim_rng`, the outcome stream. `total_gates` counts only
the **measured** phase (warmup is pre-scrambling, not part of the reported
depth).

## 4. Observables

After the loop the runner reads, based on `spec.order_parameter`:

- **half-cut entropy** $S(\{0,\dots,n/2-1\})$: *always*, for every picture.
- **code dimension** $k=S(\text{system})$: `"code_dimension"` (purification).
- **reference entropy** $S(\text{reference})\in\{0,1\}$: `"ref_entropy"`
  (single_ref).

All are rank-based, phase-exact quantities (see [`picture`](picture.ipynb)).
Each picture fills its own slot and leaves the others `None`:

In [3]:
purif = CircuitConfig(graph_spec='cycle', n=8, picture='purification', p=0.16, depth_factor=4)
sref  = CircuitConfig(graph_spec='cycle', n=8, picture='single_ref', p=0.16, depth_factor=4)
rp = simulate(purif, sample_seed=3)
rr = simulate(sref, sample_seed=3)
rs = simulate(cfg, sample_seed=3)
print('purification: code_dimension =', rp.code_dimension, '| ref_entropy =', rp.ref_entropy, '| half =', rp.entropy_half_cut)
print('single_ref  : code_dimension =', rr.code_dimension, '| ref_entropy =', rr.ref_entropy, '| half =', rr.entropy_half_cut)
print('pure_state  : code_dimension =', rs.code_dimension, '| ref_entropy =', rs.ref_entropy, '| half =', rs.entropy_half_cut)


purification: code_dimension = 0 | ref_entropy = None | half = 2
single_ref  : code_dimension = None | ref_entropy = 0 | half = 2
pure_state  : code_dimension = None | ref_entropy = None | half = 2


## 5. The `SampleRecord` union shape

Every picture returns the **same** dataclass; picture-specific observables
are `None` when they don't apply. So a mixed batch loads under one schema
without branching on the picture. The contract for the future
`sparsegf2.analysis` layer: **optional fields may be added later; required
fields may not.** Required (identity + diagnostics) vs optional (observables,
runtime, side-payload):

In [4]:
import dataclasses
from sparsegf2.circuits import SampleRecord
req = [f.name for f in dataclasses.fields(SampleRecord) if f.default is dataclasses.MISSING]
opt = [f.name for f in dataclasses.fields(SampleRecord) if f.default is not dataclasses.MISSING]
print('required:', req)
print('optional:', opt)


required: ['sample_seed', 'picture', 'total_layers', 'total_gates', 'total_measurements', 'gate_to_meas_ratio_expected', 'gate_to_meas_ratio_actual']
optional: ['code_dimension', 'ref_entropy', 'entropy_half_cut', 'runtime_total_s', 'final_tableau']


## 6. Reproducibility, injected tables, `save_tableau`

Same `(config, sample_seed)` → identical record (both streams are seeded
deterministically). `save_tableau=True` attaches the full
`to_symplectic()` $[X|Z]$ matrix, shape $(2N, 2N)$ for $N$ physical qubits.
And a custom Clifford table can be injected without monkeypatching:

In [5]:
a = simulate(cfg, sample_seed=0); b = simulate(cfg, sample_seed=0)
print('reproducible:', (a.total_measurements, a.entropy_half_cut) == (b.total_measurements, b.entropy_half_cut))
rt = simulate(purif, sample_seed=0, save_tableau=True)
print('purification final_tableau shape:', rt.final_tableau.shape, '= (2*2n, 2*2n) for N=16')
from sparsegf2.circuits import SimulationRunner
from sparsegf2.circuits._clifford_table import sp4_table
runner = SimulationRunner(cfg, clifford_table=sp4_table().copy())
print('injected-table run total_gates:', runner.run(0).total_gates)


reproducible: True
purification final_tableau shape: (32, 32) = (2*2n, 2*2n) for N=16
injected-table run total_gates: 128


## Summary

- `simulate` / `SimulationRunner` execute one realization into a
  `SampleRecord`.
- **Two independent RNG streams**: construction (`base+sample`) and outcome
  (`[base+sample, KEY]`), decouple the circuit from its measurement coins,
  with a seed trick that avoids cross-sample collisions.
- Observables are the phase-exact rank quantities; the record has a union
  shape so analysis sees one schema.

This is the top of the package. For the architecture and the MIPT framing,
see the [package overview](overview.ipynb).